# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

# Print summary
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's examine what record sets, fields, and columns are available in the dataset. This is necessary in Croissant datasets, as all references must use the unique `@id`.

In [ ]:
# Find all record sets (entities with '@type' == 'RecordSet') in metadata
record_sets = []
if 'recordSet' in metadata:
    for rs in metadata['recordSet']:
        if isinstance(rs, dict):
            record_sets.append(rs['@id'])
        else:
            record_sets.append(rs)

if not record_sets:
    # Sometimes Croissant datasets don't use 'recordSet' at top level; try a search
    def collect_record_sets(obj):
        sets = []
        if isinstance(obj, dict):
            if obj.get('@type') == 'RecordSet' and '@id' in obj:
                sets.append(obj['@id'])
            for v in obj.values():
                sets += collect_record_sets(v)
        elif isinstance(obj, list):
            for item in obj:
                sets += collect_record_sets(item)
        return sets
    record_sets = collect_record_sets(metadata)

# Print all record sets with their IDs
print('Record sets:')
for record_set in record_sets:
    print(f'- {record_set}')

# Display fields/columns for one record set
if record_sets:
    # Get the full record set definition
    def find_by_id(obj, search_id):
        if isinstance(obj, dict):
            if obj.get('@id') == search_id:
                return obj
            for v in obj.values():
                found = find_by_id(v, search_id)
                if found:
                    return found
        elif isinstance(obj, list):
            for item in obj:
                found = find_by_id(item, search_id)
                if found:
                    return found
        return None

    rs_def = find_by_id(metadata, record_sets[0])
    print(f"\nFields/columns in record set {record_sets[0]}:")
    if rs_def and 'field' in rs_def:
        for f in rs_def['field']:
            if isinstance(f, dict):
                print(f"- {f['@id']} ({f.get('name','')})")
            else:
                print(f"- {f}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set
dataframes = {}
print("\nLoading data from record sets:")
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded record set {record_set} with {len(df)} rows.")
    except Exception as e:
        print(f"Could not load {record_set}: {e}")

# Display column names of first loaded record set
if record_sets:
    rs_id = record_sets[0]
    if rs_id in dataframes:
        print("\nColumns in first record set:")
        print(dataframes[rs_id].columns.tolist())
        print("\nSample records:")
        display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a suitable numeric field for analysis. We'll use the field and column `@id`s as referenced from the overview above.

In [ ]:
import numpy as np

# Choose the main record set and likely numeric fields
main_rs_id = record_sets[0] if record_sets else None
df = dataframes.get(main_rs_id, pd.DataFrame())

# Try to find numeric columns (e.g. GAD-7/PHQ-9/PSQ scores)
numeric_fields = [col for col in df.columns if df[col].dtype in [np.int64, np.float64]]

# If numeric fields not found, try to select likely fields
if not numeric_fields:
    # Try some common column names
    likely_fields = [col for col in df.columns if any(key in col.lower() for key in ['score','gad','phq','psq'])]
    numeric_fields = likely_fields

print("Numeric fields in record set:", numeric_fields)

# Select one field for filtering
if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize this field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field (try to find demographic group)
    candidate_group_fields = [col for col in df.columns if any(word in col.lower() for word in ['village', 'gender', 'age', 'education', 'marital'])]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print("Grouped mean:")
        display(grouped_df.head())
else:
    print("No suitable numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the main numeric field
if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, visualize group differences
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} distribution by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Kilifi survey provides detailed mental health scores and demographic information at the record set granularity.
- Filtering and grouping by demographic fields allows for insightful summary statistics and visualizations.
- `mlcroissant` makes it easy to reference entities by `@id`, ensuring reproducibility and clarity when working across record sets and fields.
- Further analyses can extend this workflow to model variable relationships, assess data completeness, and support actionable insights for public health research.

-- End of notebook --